## Section 1 — Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

X_train = pd.read_csv("data/processed/X_train.csv")
X_test = pd.read_csv("data/processed/X_test.csv")
y_train = pd.read_csv("data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("data/processed/y_test.csv").squeeze()
X_predict = pd.read_csv("data/processed/X_predict.csv")

print(f"X_train:   {X_train.shape}")
print(f"X_test:    {X_test.shape}")
print(f"X_predict: {X_predict.shape}")

## Section 2 — Encode Categorical Features

In [ ]:
cat_features = [
    "property_type",
    "neighbourhood",
    "requested_timeline",
    "referral_source",
    "homeowner_status",
    "preferred_contact",
    "lead_capture_weather",
    "customer_age_bracket",
    "lead_weekday",
]

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(X_train[cat_features])

X_train_enc = X_train.copy()
X_test_enc = X_test.copy()
X_predict_enc = X_predict.copy()

X_train_enc[cat_features] = encoder.transform(X_train[cat_features]).astype(int)
X_test_enc[cat_features] = encoder.transform(X_test[cat_features]).astype(int)
X_predict_enc[cat_features] = encoder.transform(X_predict[cat_features]).astype(int)

# has_pets is stored as bool/string in the CSV — cast to int so XGBoost can use it
for df in [X_train_enc, X_test_enc, X_predict_enc]:
    df["has_pets"] = df["has_pets"].astype(int)

# Note: lead_month_sin and lead_month_cos are cyclical encodings of the original
# lead_month feature (sin/cos transform). They flow through as numeric automatically.

print("Encoded training set sample:")
X_train_enc.head()

## Section 3 — Train XGBoost

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss",
    early_stopping_rounds=50,
    verbosity=0,
)

model.fit(
    X_train_enc,
    y_train_enc,
    eval_set=[(X_test_enc, y_test_enc)],
    verbose=100,
)

## Section 4 — Evaluate on Test Set

In [ ]:
y_pred_enc = model.predict(X_test_enc)
y_pred = le.inverse_transform(y_pred_enc)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f} ({acc * 100:.1f}%)")
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

## Section 5 — Confusion Matrix

In [ ]:
labels = ["Low", "Medium", "High"]
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", colorbar=False, ax=ax)
ax.set_title("XGBoost \u2014 Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig("figures/10_xgboost_confusion_matrix.png", dpi=150)
plt.show()

## Section 6 — Feature Importance

In [ ]:
importances = model.feature_importances_
feature_names = X_train_enc.columns

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.plot.barh(ax=ax, color="#55A868")
ax.set_title("XGBoost \u2014 Feature Importance", fontsize=14, fontweight="bold")
ax.set_xlabel("Importance")
plt.tight_layout()
fig.savefig("figures/11_xgboost_feature_importance.png", dpi=150)
plt.show()

print("\nFeature importances (descending):")
for name, imp in feat_imp.sort_values(ascending=False).items():
    print(f"  {name:30s} {imp:.4f}")

## Section 7 — Predict on Unlabeled Leads

In [ ]:
preds_enc = model.predict(X_predict_enc)
preds = le.inverse_transform(preds_enc)
proba = model.predict_proba(X_predict_enc)

results = X_predict.copy()
results["predicted_profit_band"] = preds

for i, cls in enumerate(le.classes_):
    results[f"prob_{cls}"] = proba[:, i]

results.to_csv("xgboost_predictions.csv", index=False)
print(f"Saved xgboost_predictions.csv ({len(results)} rows)")
print()
print("Predicted profit band distribution:")
print(results["predicted_profit_band"].value_counts())